In [2]:
# Khai báo thư viện và định nghĩa hàm tính WOE/IV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import scipy.stats as stats

# Định nghĩa hàm tính toán WOE và IV cho một biến liên tục
def calculate_woe_iv_continuous(df, feature, target, num_bins=5):
    # Copy dữ liệu để tránh ghi đè
    df_temp = df[[feature, target]].copy()
    
    # Chia khoảng bằng pd.qcut (đảm bảo số lượng hồ sơ mỗi bin tương đương nhau)
    df_temp['bin'] = pd.qcut(df_temp[feature], q=num_bins, duplicates='drop')
    
    # Gom nhóm để tính tổng Good và Bad
    # Giả định: target = 1 là Bad (Nợ xấu), target = 0 là Good (Trả tốt)
    total_bads = df_temp[target].sum()
    total_goods = df_temp[target].count() - total_bads
    
    grouped = df_temp.groupby('bin', observed=False).agg(
        Good=(target, lambda x: (x == 0).sum()),
        Bad=(target, 'sum'),
        Total=(target, 'count')
    ).reset_index()
    
    # Tính toán tỷ lệ phần trăm
    grouped['%_Good'] = grouped['Good'] / total_goods
    grouped['%_Bad'] = grouped['Bad'] / total_bads
    
    # Sửa lỗi chia cho 0 hoặc log(0) nếu có bin nào đó không có Good hoặc Bad
    grouped['%_Good'] = grouped['%_Good'].replace(0, 0.0001)
    grouped['%_Bad'] = grouped['%_Bad'].replace(0, 0.0001)
    
    # Công thức toán học của WOE và IV
    grouped['WOE'] = np.log(grouped['%_Good'] / grouped['%_Bad'])
    grouped['IV_bin'] = (grouped['%_Good'] - grouped['%_Bad']) * grouped['WOE']
    
    # Tổng hợp Information Value (IV) của toàn bộ biến
    total_iv = grouped['IV_bin'].sum()
    grouped['bin'] = grouped['bin'].astype(str)
    
    return grouped, total_iv

In [3]:
# Đọc dữ liệu và lọc biến WOE chiến lược 
# Đọc file dữ liệu của bạn
df = pd.read_csv('../Data/Risk_data_FE.csv',index_col=0)
target_col = 'loan_status'

# Sạch hóa dữ liệu bị khuyết thiếu trước khi xử lý
df_clean = df.dropna(subset=[target_col, 'loan_int_rate', 'debt_to_income_ratio', 'installment_income_ratio']).copy()

# Khởi tạo một dictionary để lưu bảng tra cứu WOE của các biến
woe_maps = {}
iv_summary = {}

features_to_woe = ['loan_int_rate', 'debt_to_income_ratio', 'installment_income_ratio']

for feat in features_to_woe:
    woe_table, iv_value = calculate_woe_iv_continuous(df_clean, feat, target_col, num_bins=6)
    woe_maps[feat] = woe_table
    iv_summary[feat] = iv_value
    
    print(f"=== BẢNG WOE & IV CỦA BIẾN: {feat} (Tổng IV = {iv_value:.4f}) ===")
    print(woe_table[['bin', 'Total', 'Good', 'Bad', 'WOE']])
    print("-" * 60)

=== BẢNG WOE & IV CỦA BIẾN: loan_int_rate (Tổng IV = 0.7429) ===
              bin  Total  Good   Bad       WOE
0   (5.419, 7.49]   5967  5490   477  1.150741
1    (7.49, 9.62]   4627  4086   541  0.729476
2   (9.62, 10.99]   5796  4876   920  0.375280
3  (10.99, 12.68]   4795  3952   843  0.252584
4  (12.68, 14.27]   5321  4106  1215 -0.074721
5  (14.27, 23.22]   5173  2344  2829 -1.480491
------------------------------------------------------------
=== BẢNG WOE & IV CỦA BIẾN: debt_to_income_ratio (Tổng IV = 0.5314) ===
               bin  Total  Good   Bad       WOE
0  (0.0635, 0.219]   5280  4678   602  0.757942
1    (0.219, 0.28]   5280  4604   676  0.626061
2    (0.28, 0.333]   5280  4549   731  0.535823
3   (0.333, 0.388]   5279  4362   917  0.267152
4   (0.388, 0.465]   5280  3981  1299 -0.172488
5   (0.465, 1.054]   5280  2680  2600 -1.262121
------------------------------------------------------------
=== BẢNG WOE & IV CỦA BIẾN: installment_income_ratio (Tổng IV = 0.3753) ===


In [4]:
# Ánh xạ điểm WOE vào tập dữ liệu (WOE Transformation)
df_woe = pd.DataFrame()
df_woe[target_col] = df_clean[target_col]

# Thực hiện ánh xạ (Mapping) giá trị thô sang điểm WOE
for feat in features_to_woe:
    df_clean['bin_temp'] = pd.qcut(df_clean[feat], q=6, duplicates='drop').astype(str)
    
    # Tạo dictionary để map từ tên bin sang giá trị WOE
    mapping_dict = dict(zip(woe_maps[feat]['bin'], woe_maps[feat]['WOE']))
    
    df_woe[f'{feat}_WOE'] = df_clean['bin_temp'].map(mapping_dict)

print("Kích thước ma trận dữ liệu WOE mới:", df_woe.shape)
print(df_woe.head())

Kích thước ma trận dữ liệu WOE mới: (31679, 4)
   loan_status  loan_int_rate_WOE  debt_to_income_ratio_WOE  \
1            0           0.252584                  0.626061   
2            1          -0.074721                 -1.262121   
3            1          -1.480491                 -1.262121   
4            1          -0.074721                 -1.262121   
5            1           1.150741                 -1.262121   

   installment_income_ratio_WOE  
1                      0.424459  
2                     -0.968562  
3                     -0.968562  
4                     -0.968562  
5                     -0.325427  


In [5]:
# Huấn luyện mô hình Logistic Regression trên nền WOE
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_woe = df_woe.drop(columns=[target_col])
y_woe = df_woe[target_col]

# 1. Kiểm tra VIF trên dữ liệu WOE
vif_data = pd.DataFrame()
vif_data["Feature"] = X_woe.columns
vif_data["VIF"] = [variance_inflation_factor(X_woe.values, i) for i in range(X_woe.shape[1])]
print("=== HỆ SỐ VIF MỚI TRÊN NỀN BIẾN ĐỔI WOE ===")
print(vif_data)
print("-" * 60)

# 2. Chia tập Train/Test
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_woe, y_woe, test_size=0.3, random_state=42, stratify=y_woe)

# 3. Fit mô hình Logistic Regression
log_model_woe = LogisticRegression(random_state=42)
log_model_woe.fit(X_train_w, y_train_w)

print(f"Hệ số chặn (Intercept Beta_0): {log_model_woe.intercept_[0]:.4f}")
for feat, coef in zip(X_woe.columns, log_model_woe.coef_[0]):
    print(f"Trọng số (Beta) của {feat}: {coef:.4f}")

=== HỆ SỐ VIF MỚI TRÊN NỀN BIẾN ĐỔI WOE ===
                        Feature       VIF
0             loan_int_rate_WOE  1.019203
1      debt_to_income_ratio_WOE  1.551684
2  installment_income_ratio_WOE  1.549001
------------------------------------------------------------
Hệ số chặn (Intercept Beta_0): -1.2990
Trọng số (Beta) của loan_int_rate_WOE: -1.0207
Trọng số (Beta) của debt_to_income_ratio_WOE: -0.8113
Trọng số (Beta) của installment_income_ratio_WOE: -0.4994


In [6]:
# Đánh giá hiệu năng và tìm cutoff của Mô hình WOE mới
y_pred_prob_w = log_model_woe.predict_proba(X_test_w)[:, 1]

# Tính AUC và Gini
auc_w = roc_auc_score(y_test_w, y_pred_prob_w)
gini_w = 2 * auc_w - 1

# Tính chỉ số KS
data_ks_w = pd.DataFrame({'real': y_test_w, 'prob': y_pred_prob_w})
bads_w = data_ks_w[data_ks_w['real'] == 1]['prob']
goods_w = data_ks_w[data_ks_w['real'] == 0]['prob']
ks_stat_w, _ = stats.ks_2samp(bads_w, goods_w)

# Tìm điểm Cutoff tối ưu
fpr_w, tpr_w, thresholds_w = roc_curve(y_test_w, y_pred_prob_w)
optimal_idx_w = np.argmax(tpr_w - fpr_w)
optimal_cutoff_w = thresholds_w[optimal_idx_w]

print("======================================================")
print(" CHỈ SỐ HIỆU NĂNG CỦA MÔ HÌNH DÙNG PHƯƠNG PHÁP WOE")
print("======================================================")
print(f"[*] AUC Score      : {auc_w:.4f}")
print(f"[*] Gini Score     : {gini_w:.4f}")
print(f"[*] KS Statistic   : {ks_stat_w:.4f}")
print(f"[👉] CUTOFF PD TỐI ƯU ĐỀ XUẤT: {optimal_cutoff_w:.4f}")
print("======================================================")

 CHỈ SỐ HIỆU NĂNG CỦA MÔ HÌNH DÙNG PHƯƠNG PHÁP WOE
[*] AUC Score      : 0.8192
[*] Gini Score     : 0.6384
[*] KS Statistic   : 0.5242
[👉] CUTOFF PD TỐI ƯU ĐỀ XUẤT: 0.2576


In [7]:
import pandas as pd
import numpy as np

def calculate_dataset_iv(df, target, num_bins=5):
    iv_records = []
    
    # Lấy danh sách tất cả các cột trừ cột mục tiêu (target)
    features = [col for col in df.columns if col != target]
    
    # Tính tổng số ca Bad và Good của toàn bộ dataset
    total_bads = df[target].sum()
    total_goods = df[target].count() - total_bads
    
    for feat in features:
        try:
            # Tạo bản sao dữ liệu nhỏ để tính toán
            df_temp = df[[feat, target]].copy().dropna()
            
            # Kiểm tra nếu là biến số (liên tục) thì chia khoảng, nếu là biến chữ (phân loại) thì giữ nguyên nhóm
            if np.issubdtype(df_temp[feat].dtype, np.number):
                df_temp['bin'] = pd.qcut(df_temp[feat], q=num_bins, duplicates='drop')
            else:
                df_temp['bin'] = df_temp[feat]
                
            # Gom nhóm để đếm số lượng Good và Bad trong từng nhóm (bin)
            grouped = df_temp.groupby('bin', observed=False).agg(
                Good=(target, lambda x: (x == 0).sum()),
                Bad=(target, 'sum')
            ).reset_index()
            
            # Tính tỷ lệ % Good và % Bad của từng bin so với tổng thể
            grouped['%_Good'] = grouped['Good'] / total_goods
            grouped['%_Bad'] = grouped['Bad'] / total_bads
            
            # Xử lý tránh lỗi chia cho 0 hoặc log(0)
            grouped['%_Good'] = grouped['%_Good'].replace(0, 0.0001)
            grouped['%_Bad'] = grouped['%_Bad'].replace(0, 0.0001)
            
            # Công thức tính WOE và IV của từng bin
            grouped['WOE'] = np.log(grouped['%_Good'] / grouped['%_Bad'])
            grouped['IV_bin'] = (grouped['%_Good'] - grouped['%_Bad']) * grouped['WOE']
            
            # Tổng hợp IV của toàn bộ biến số
            total_iv = grouped['IV_bin'].sum()
            
            # Phân loại sức mạnh của biến theo quy chuẩn ngành rủi ro
            if total_iv < 0.02:
                predictive_power = "1. Quá yếu (Loại bỏ)"
            elif total_iv < 0.1:
                predictive_power = "2. Yếu (Cân nhắc)"
            elif total_iv < 0.3:
                predictive_power = "3. Trung bình (Tốt)"
            elif total_iv < 0.5:
                predictive_power = "4. Mạnh (Cốt lõi)"
            else:
                predictive_power = "5. Quá mạnh (Nghi ngờ Leakage)"
                
            iv_records.append({
                'Tên Biến (Feature)': feat,
                'Giá trị IV (Information Value)': round(total_iv, 4),
                'Sức mạnh dự báo (Predictive Power)': predictive_power
            })
            
        except Exception as e:
            # Bỏ qua nếu gặp lỗi ép kiểu dữ liệu đặc biệt
            continue
            
    # Chuyển kết quả thành DataFrame để sắp xếp đẹp mắt
    df_iv = pd.DataFrame(iv_records)
    df_iv = df_iv.sort_values(by='Giá trị IV (Information Value)', ascending=False).reset_index(drop=True)
    
    return df_iv

# --- CHẠY THỬ VÀ IN BẢNG IV ---
# Sửa 'loan_status' nếu tên cột mục tiêu trong file 'Rick_data_FE.csv' của bạn khác đi
df_raw = pd.read_csv('../Data/Risk_data_FE.csv')
target_column = 'loan_status'

print("================================================================")
print("     BẢNG XẾP HẠNG SỨC MẠNH BIẾN SỐ TÍN DỤNG (INFORMATION VALUE) ")
print("================================================================")

df_iv_result = calculate_dataset_iv(df_raw, target_column, num_bins=6)
# Hiển thị toàn bộ bảng kết quả
pd.set_option('display.max_rows', None)
print(df_iv_result)

     BẢNG XẾP HẠNG SỨC MẠNH BIẾN SỐ TÍN DỤNG (INFORMATION VALUE) 
            Tên Biến (Feature)  Giá trị IV (Information Value)  \
0           loan_grade_numeric                          0.8302   
1         loan_to_income_ratio                          0.7799   
2          loan_percent_income                          0.7786   
3                loan_int_rate                          0.7429   
4         debt_to_income_ratio                          0.5314   
5                person_income                          0.4338   
6     installment_income_ratio                          0.3753   
7                   other_debt                          0.2388   
8    cb_person_default_on_file                          0.1688   
9             risk_interaction                          0.1562   
10                   loan_amnt                          0.0825   
11           person_emp_length                          0.0605   
12              emp_length_num                          0.0605   
13        

Tiêu chuẩn chỉ số IV (Information value):
- 0.02 <= IV < 0.1: Sức mạnh yếu, biến có một chút khả năng dự báo nhưng không nhiều (Cân nhắc giữ lại nếu thiếu biến)
- 0.1<=IV<0.3: Sức mạnh trung bình, biến rất tốt, có lực phân tách rủi ro rõ ràng (bắt buộc giữ lại để train mô hình)
- 0.3<=IV<0.5: Sức mạnh rất mạnh, đây là các biến vàng, động lực chính của mô hình rủi ro 
- IV >= 0.5: Quá mạnh, cần nghi ngờ: Nếu một biến có IV tăng vọt lên trên 0.5 hoặc 0.7, nên đặt câu hỏi nghi ngờ liệu có bị dính lỗi Data leakage (rò rỉ dữ liệu), túc là nó vô tình chứa kết quả tương lai

- loan_grade_numeric : 0.8302 Bị Dataleakage. Hệ thống chấm điểm này đã gom tất cả các thông tin rủi ro nhất của khách hàng để quy ra một nhãn dán A, B..., Khi đưa vào mô hình, mô hình logistic regression sẽ lười biếng, nó chỉ cần nhìn vào loan_grade là đoán được ngay loan_status mà bỏ qua việc học từ các hành vi tài chính gốc của ngân hàng => Không còn yếu tố dự đoán rủi ro => Xóa bỏ

- Loan_to_income_ratio: 0.7799 + Loan_percen_income: 0.7786: Dính đa cộng tuyến nặng, bản chất hai biến này là một. Khi một bộ dữ liệu có hai biến mang cùng một thông tin toán học, chúng sẽ bị đa cộng tuyến hoàn hảo. Thêm hai biến này vào mô hình sẽ khiếm VIF bị nổ tung, vọt lên ngưỡng an toàn => Chỉ lấy 1 trong 2

- Loan_int_rate: 0.7429: Thông tin vàng hay là data leakage ? Biến lãi xuất có IV = 0.74 là một trường hợp điển hình cần phải xem xét kỹ cơ chế vận hành của tổ chức tín dụng
    + Trường hợp 1: Dính Data leakage, nếu ngân hàng đó áp dụng chính sách định giá giựa trên rủi ro từ trước. Nghĩa là hệ thống thấy khách hàng này có rủi ro quá cao, nên họ chủ động ép khách hàng phải chịu lãi suất cao để bù đắp cho chi phí rủi ro. Nếu rơi vào trường hợp này, biến lãi suất đã bị nhiễm thông tin rủi ro => Cân nhắc loại bỏ hoặc đưa vào nhóm WOE kiểm soát chặt.
    + Trường hợp 2: Thông tin nghiệp vụ hợp lý. Lãi suất cao chính là nguyên nhân trực tiếp dẫn đến vỡ nợ. Lãi suất càng cao thì số tiền phải trả hàng tháng càng lớn => quan hệ nhân quả => Giữ lại và biến đổi WOE

Thực hiện điều chỉnh IV, WOE trong file 05.4 Modeling_03_Fix.ipynb

